# Real Estate Research Agent

This notebook combines **Milvus semantic search, Neo4j structured
retrieval, CSV listing enrichment, and Claude-generated answers**.

## How it works

1. Claude interprets the question, selects a retrieval strategy,
   and normalizes filters using values discovered from the database.
2. Milvus finds semantically relevant property IDs for descriptive queries.
3. Neo4j retrieves authoritative property information and applies filters.
4. CSV data supplies listing URLs, image URLs, and descriptions.
5. Claude generates a grounded response with clickable listings.
6. Gradio provides a simple interface for testing.

## 1. Install dependencies and connect services


In [17]:
import os
import re
import json
import ast
import math
import glob
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
from anthropic import Anthropic
from neo4j import GraphDatabase
from pymilvus import MilvusClient
from fastembed import TextEmbedding
from IPython.display import display

load_dotenv()

def env_required(name, *alternatives):
    for key in (name, *alternatives):
        value = os.getenv(key)
        if value:
            return value
    raise ValueError(
        f"Missing {name}. Add it to .env or your environment."
    )

ANTHROPIC_API_KEY = env_required("ANTHROPIC_API_KEY")
NEO4J_URI = env_required("NEO4J_URI")
NEO4J_USERNAME = env_required("NEO4J_USERNAME", "NEO4J_USER")
NEO4J_PASSWORD = env_required("NEO4J_PASSWORD")
MILVUS_URI = env_required("MILVUS_URI")

ANTHROPIC_MODEL = os.getenv(
    "ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"
)
MILVUS_COLLECTION = os.getenv(
    "MILVUS_COLLECTION", "property_listings"
)
EMBEDDING_MODEL_NAME = os.getenv(
    "EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5"
)
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
TOP_K = int(os.getenv("RETRIEVAL_TOP_K", "8"))
MILVUS_CANDIDATE_LIMIT = int(
    os.getenv("MILVUS_CANDIDATE_LIMIT", "30")
)
PROPERTY_CSV_PATH = os.getenv("PROPERTY_CSV_PATH", "")

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)

neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)
neo4j_driver.verify_connectivity()

milvus_kwargs = {"uri": MILVUS_URI}
if os.getenv("MILVUS_TOKEN"):
    milvus_kwargs["token"] = os.getenv("MILVUS_TOKEN")

milvus_client = MilvusClient(**milvus_kwargs)
embedding_model = TextEmbedding(model_name=EMBEDDING_MODEL_NAME)

print("Connections initialized.")
print("Claude model:", ANTHROPIC_MODEL)
print("Milvus collection:", MILVUS_COLLECTION)

Connections initialized.
Claude model: claude-haiku-4-5-20251001
Milvus collection: property_listings


## 2. Load and inspect the property CSV

In [2]:
def locate_property_csv():
    if PROPERTY_CSV_PATH:
        path = Path(PROPERTY_CSV_PATH).expanduser()
        if not path.exists():
            raise FileNotFoundError(f"CSV not found: {path}")
        return path

    candidates = []
    for pattern in (
        "data/**/*.csv",
        "datasets/**/*.csv",
        "*.csv",
        "../data/**/*.csv",
    ):
        candidates.extend(Path(".").glob(pattern))

    candidates = list(dict.fromkeys(
        p.resolve() for p in candidates if p.is_file()
    ))

    if len(candidates) == 1:
        return candidates[0]

    if not candidates:
        raise FileNotFoundError(
            "No CSV found. Set PROPERTY_CSV_PATH in .env."
        )

    raise ValueError(
        "Multiple CSV files found. Set PROPERTY_CSV_PATH in .env:\n"
        + "\n".join(map(str, candidates[:20]))
    )

csv_path = locate_property_csv()
properties_df = pd.read_csv(csv_path, low_memory=False)
properties_df.columns = [
    str(c).strip() for c in properties_df.columns
]

print("CSV:", csv_path)
print("Rows:", len(properties_df))
print("Columns:", len(properties_df.columns))
display(pd.DataFrame({
    "column": properties_df.columns,
    "non_null": [
        properties_df[c].notna().sum()
        for c in properties_df.columns
    ],
}))

CSV: /home/ekkel-ai/momina/projects/real-estate-project/data/processed/property_listings_raw.csv
Rows: 6495
Columns: 57


,column,non_null
0,id,6495
1,crm_id,6495
2,strapi_id,6495
3,slug,6495
4,title,6495
5,listing_type,6495
6,department,6495
7,search_type,6495
8,status,6495
9,price,6495


## 3. Normalize IDs and discover database vocabulary

Discover the actual values used for listing type, property type,
status, and community from Neo4j. The planner uses these values to avoid
mismatches such as asking for `sale` when the database stores `purchase`.

In [3]:
def normalize_id(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    value = str(value).strip()
    if not value or value.lower() in {"nan", "none", "null"}:
        return None

    # Normalize numeric IDs stored as values such as 123.0.
    if re.fullmatch(r"\d+\.0+", value):
        value = value.split(".")[0]
    return value


def find_column(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None


CSV_ID_COLUMN = find_column(
    properties_df,
    ["property_id", "id", "crm_id", "strapi_id", "listing_id"]
)

if not CSV_ID_COLUMN:
    raise ValueError(
        "Could not identify the CSV property ID column. "
        "Inspect the columns above and update CSV_ID_COLUMN."
    )

properties_df["_join_id"] = properties_df[CSV_ID_COLUMN].map(
    normalize_id
)
properties_df = properties_df.dropna(subset=["_join_id"])
properties_df = properties_df.drop_duplicates(
    subset=["_join_id"], keep="first"
)
properties_df = properties_df.set_index("_join_id", drop=False)

def run_cypher(query, parameters=None):
    with neo4j_driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]


db_counts = run_cypher("""
    MATCH (p:Property)
    RETURN count(p) AS property_count
""")
print("Neo4j property count:", db_counts[0]["property_count"])

vocabulary_rows = run_cypher("""
    MATCH (p:Property)
    OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
    RETURN
        collect(DISTINCT toString(p.listing_type)) AS listing_types,
        collect(DISTINCT toString(p.property_type)) AS property_types,
        collect(DISTINCT toString(p.status)) AS statuses,
        collect(DISTINCT toString(c.name)) AS communities
""")

vocabulary = vocabulary_rows[0] if vocabulary_rows else {}

def clean_vocabulary(values):
    return sorted({
        str(v).strip()
        for v in (values or [])
        if v and str(v).strip().lower() not in
        {"nan", "none", "null"}
    })

vocabulary = {
    key: clean_vocabulary(value)
    for key, value in vocabulary.items()
}

for key, values in vocabulary.items():
    print(f"{key}: {len(values)} distinct values")
    print(values[:20])

print("CSV unique property IDs:", len(properties_df))

Neo4j property count: 6495
listing_types: 2 distinct values
['purchase', 'rental']
property_types: 9 distinct values
['apartment', 'labour camp', 'land', 'office space', 'retail', 'townhouse', 'villa', 'warehouse', 'whole building']
statuses: 4 distinct values
['For Sale', 'Let', 'Sold', 'To Let']
communities: 230 distinct values
['(Akoya By DAMAC)', '(DUB221897)', 'AKOYA Oxygen', 'Akoya', 'Akoya By DAMAC', 'Al Badaa', 'Al Barari', 'Al Barsha', 'Al Barsha 3', 'Al Fujan', 'Al Furjan', 'Al Furjan West', 'Al Jaddaf', 'Al Khawaneej', 'Al Kifaf', 'Al Safa', 'Al Satwa', 'Al Sufouh', 'Al Waha Villas', "Al Warqa'a"]
CSV unique property IDs: 6495


## 4. One LLM planning call

Claude returns one structured plan. It chooses the retrieval strategy,
extracts filters, and normalizes requested values against the actual
database vocabulary.

Strategy choices:
- `semantic`: descriptive or similarity-based requests
- `structured`: explicit filters or exact property lookup
- `hybrid`: filters plus descriptive preferences
- `analytics`: aggregate or community-level questions

In [23]:
PLANNER_PROMPT = """
You are the query planner for a real-estate research system.

Return ONLY valid JSON. Do not answer the user's property question.

Database vocabulary:
{vocabulary}

Return this schema:
{{
  "query_type": "property_search|exact_lookup|community_analysis|portfolio_analysis|general_real_estate",
  "retrieval_strategy": "semantic|structured|hybrid|analytics",
  "normalized_query": "clear normalized search request",
  "semantic_query": "description used for semantic similarity",
  "filters": {{
    "community": null,
    "property_type": null,
    "transaction_type": null,
    "status": null,
    "min_price": null,
    "max_price": null,
    "min_beds": null,
    "max_beds": null,
    "min_bathrooms": null,
    "max_bathrooms": null
  }},
  "exact_property_id": null,
  "community": null,
  "intent_preferences": [],
  "answer_mode": "listings|summary|analytics|explanation",
  "clarification_needed": false,
  "clarification_question": null
}}

Rules:
1. Use only vocabulary values that actually exist in the supplied
   database vocabulary for community, property type, transaction type,
   and status. Match user wording to the closest actual value.
2. For example, if the user says "sale" but the database vocabulary
   uses "purchase", use "purchase". Do not invent a new category.
3. A property with a requested minimum or maximum must satisfy it.
4. Use structured for exact IDs and mostly explicit filters.
5. Use semantic for descriptive requests such as natural light,
   peaceful surroundings, or lifestyle fit without hard filters.
6. Use hybrid when hard filters and descriptive preferences both matter.
7. Use analytics for counts, averages, distributions, or community
   comparisons that need database aggregation.
8. Put the original descriptive intent in semantic_query.
9. Unknown numeric filters must be null, not guessed.
10. Do not invent IDs, facts, communities, or listing details.
11. Do not ask for rent/purchase, budget, community, or property type
    just because the user omitted them. These are optional filters.
12. For descriptive requests such as peaceful, bright, quiet, or
    family-friendly homes, proceed with semantic search.
13. Ask a clarification question only when the request cannot
    reasonably be interpreted or searched without clarification.
"""


def parse_json_response(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def plan_query(user_query):
    prompt = PLANNER_PROMPT.format(
        vocabulary=json.dumps(vocabulary, ensure_ascii=False)
    )
    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1200,
        system=prompt,
        messages=[{"role": "user", "content": user_query}],
    )
    raw = "".join(
        block.text for block in response.content
        if getattr(block, "type", None) == "text"
    )
    plan = parse_json_response(raw)

    required_keys = {
        "query_type", "retrieval_strategy", "filters",
        "normalized_query", "semantic_query"
    }
    missing = required_keys - set(plan)
    if missing:
        raise ValueError(f"Planner response missing keys: {missing}")

    if plan["retrieval_strategy"] not in {
        "semantic", "structured", "hybrid", "analytics"
    }:
        raise ValueError("Planner returned an unsupported strategy.")

    return plan


example_plan = plan_query(
    "Show apartments in Dubai Marina under 250k AED"
)
display(example_plan)

{'query_type': 'property_search',
 'retrieval_strategy': 'structured',
 'normalized_query': 'Find apartments in Dubai Marina with maximum price of 250000 AED',
 'semantic_query': None,
 'filters': {'community': 'Dubai Marina',
  'property_type': 'apartment',
  'transaction_type': None,
  'status': None,
  'min_price': None,
  'max_price': 250000,
  'min_beds': None,
  'max_beds': None,
  'min_bathrooms': None,
  'max_bathrooms': None},
 'exact_property_id': None,
 'community': 'Dubai Marina',
 'intent_preferences': [],
 'answer_mode': 'listings',
 'clarification_needed': False,
 'clarification_question': None}

In [9]:
plan = plan_query("Find a peaceful home with lots of natural light")
display(plan)

{'query_type': 'property_search',
 'retrieval_strategy': 'semantic',
 'normalized_query': 'Find a peaceful home with lots of natural light',
 'semantic_query': 'peaceful serene home with abundant natural light bright airy living space',
 'filters': {'community': None,
  'property_type': None,
  'transaction_type': None,
  'status': None,
  'min_price': None,
  'max_price': None,
  'min_beds': None,
  'max_beds': None,
  'min_bathrooms': None,
  'max_bathrooms': None},
 'exact_property_id': None,
 'community': None,
 'intent_preferences': ['peaceful_environment',
  'natural_light',
  'bright_airy_spaces'],
 'answer_mode': 'listings',
 'clarification_needed': False,
 'clarification_question': None}

## 5. Retrieval helpers

The embedding model converts descriptive queries into vectors.
Milvus returns candidate property IDs. Neo4j retrieves facts by ID.

In [10]:
def embed_query(query):
    vectors = list(embedding_model.query_embed(query))
    if not vectors:
        raise ValueError("Embedding model returned no vector.")
    return vectors[0].tolist() if hasattr(vectors[0], "tolist") else list(vectors[0])


def milvus_search_ids(query, limit=None):
    limit = limit or MILVUS_CANDIDATE_LIMIT
    vector = embed_query(query)

    results = milvus_client.search(
        collection_name=MILVUS_COLLECTION,
        data=[vector],
        limit=limit,
        output_fields=[
            "property_id", "title", "price", "beds",
            "bathrooms", "community", "property_type",
            "listing_type", "status"
        ],
    )

    hits = results[0] if results else []
    output = []
    for rank, hit in enumerate(hits, start=1):
        entity = hit.get("entity", {}) or {}
        pid = normalize_id(
            entity.get("property_id", hit.get("id"))
        )
        if pid:
            output.append({
                "property_id": pid,
                "semantic_score": hit.get("distance"),
                "semantic_rank": rank,
            })
    return output


def neo4j_records_by_ids(ids):
    ids = list(dict.fromkeys(
        normalize_id(pid) for pid in ids if normalize_id(pid)
    ))
    if not ids:
        return []

    return run_cypher("""
        MATCH (p:Property)
        WHERE toString(p.property_id) IN $ids
        OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
        OPTIONAL MATCH (p)-[:IN_BUILDING]->(b:Building)
        OPTIONAL MATCH (p)-[:HAS_TYPE]->(pt:PropertyType)
        RETURN
            toString(p.property_id) AS property_id,
            p.title AS title,
            p.price AS price,
            p.beds AS beds,
            p.bathrooms AS bathrooms,
            coalesce(c.name, p.community) AS community,
            coalesce(pt.name, p.property_type) AS property_type,
            p.listing_type AS listing_type,
            p.status AS status,
            p.description AS description,
            p.latitude AS latitude,
            p.longitude AS longitude,
            b.name AS building
    """, {"ids": ids})


def numeric_value(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    try:
        cleaned = str(value).replace(",", "").strip()
        number = float(cleaned)
        return number if math.isfinite(number) else None
    except (TypeError, ValueError):
        return None


def parse_image_urls(value):
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    if isinstance(value, (list, tuple)):
        items = value
    else:
        raw = str(value).strip()
        if not raw or raw.lower() in {"nan", "none", "null", "[]"}:
            return []
        items = None
        for parser in (json.loads, ast.literal_eval):
            try:
                items = parser(raw)
                break
            except (ValueError, SyntaxError, TypeError):
                continue
        if items is None:
            items = [raw]

    if isinstance(items, dict):
        items = list(items.values())
    if not isinstance(items, (list, tuple)):
        items = [items]

    urls = []
    for item in items:
        if isinstance(item, dict):
            item = (
                item.get("url") or item.get("src")
                or item.get("original") or item.get("image")
            )
        if isinstance(item, str) and item.startswith(("http://", "https://")):
            urls.append(item.strip())

    return list(dict.fromkeys(urls))


IMAGE_COLUMN = find_column(
    properties_df,
    ["images", "image_urls", "image_url", "photos", "media"]
)
LINK_COLUMN = find_column(
    properties_df,
    ["link", "url", "listing_url", "property_url",
     "web_url", "permalink"]
)
TITLE_COLUMN = find_column(
    properties_df, ["title", "name", "property_title"]
)
DESCRIPTION_COLUMN = find_column(
    properties_df,
    ["long_description", "description", "summary"]
)

print("CSV image column:", IMAGE_COLUMN)
print("CSV listing link column:", LINK_COLUMN)
print("CSV title column:", TITLE_COLUMN)
print("CSV description column:", DESCRIPTION_COLUMN)


def enrich_with_csv(records):
    enriched = []
    for record in records:
        row = dict(record)
        pid = normalize_id(row.get("property_id"))

        if pid and pid in properties_df.index:
            source = properties_df.loc[pid]

            if isinstance(source, pd.DataFrame):
                source = source.iloc[0]

            if IMAGE_COLUMN:
                row["image_urls"] = parse_image_urls(
                    source.get(IMAGE_COLUMN)
                )
            else:
                row["image_urls"] = []

            if LINK_COLUMN:
                url = source.get(LINK_COLUMN)
                row["listing_url"] = (
                    str(url).strip()
                    if pd.notna(url) and str(url).startswith(("http://", "https://"))
                    else None
                )
            else:
                row["listing_url"] = None

            if TITLE_COLUMN and not row.get("title"):
                row["title"] = source.get(TITLE_COLUMN)

            if DESCRIPTION_COLUMN and not row.get("description"):
                row["description"] = source.get(DESCRIPTION_COLUMN)
        else:
            row["image_urls"] = []
            row["listing_url"] = None

        enriched.append(row)

    return enriched

CSV image column: images
CSV listing link column: link
CSV title column: title
CSV description column: long_description


## 6. Apply hard constraints and retrieve candidates

In [ ]:
def as_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def same_text(a, b):
    return str(a or "").strip().casefold() == str(b or "").strip().casefold()


def apply_hard_filters(records, filters):
    result = []

    for row in records:
        community = row.get("community")
        ptype = row.get("property_type")
        listing_type = row.get("listing_type")
        status = row.get("status")

        requested_community = filters.get("community")
        if requested_community and not same_text(
            community, requested_community
        ):
            # Allow a partial community match only if the actual
            # database community contains the requested normalized name.
            if str(requested_community).casefold() not in str(
                community or ""
            ).casefold():
                continue

        if filters.get("property_type") and not same_text(
            ptype, filters["property_type"]
        ):
            continue

        if filters.get("transaction_type") and not same_text(
            listing_type, filters["transaction_type"]
        ):
            continue

        if filters.get("status") and not same_text(
            status, filters["status"]
        ):
            continue

        numeric_filters = [
            ("price", "min_price", "max_price"),
            ("beds", "min_beds", "max_beds"),
            ("bathrooms", "min_bathrooms", "max_bathrooms"),
        ]

        rejected = False
        for field, min_key, max_key in numeric_filters:
            minimum = numeric_value(filters.get(min_key))
            maximum = numeric_value(filters.get(max_key))

            if minimum is None and maximum is None:
                continue

            actual = numeric_value(row.get(field))
            if actual is None:
                rejected = True
                break
            if minimum is not None and actual < minimum:
                rejected = True
                break
            if maximum is not None and actual > maximum:
                rejected = True
                break

        if not rejected:
            result.append(row)

    return result


def structured_search(filters, limit=500):
    query = """
        MATCH (p:Property)
        OPTIONAL MATCH (p)-[:LOCATED_IN]->(c:Community)
        OPTIONAL MATCH (p)-[:HAS_TYPE]->(pt:PropertyType)
        WITH p, c, pt,
            toLower(toString(coalesce(c.name, p.community, ''))) AS area,
            toLower(toString(coalesce(pt.name, p.property_type, ''))) AS ptype,
            toLower(toString(coalesce(p.listing_type, ''))) AS listing,
            toLower(toString(coalesce(p.status, ''))) AS state,
            toFloat(replace(toString(p.price), ',', '')) AS price,
            toFloat(toString(p.beds)) AS beds,
            toFloat(toString(p.bathrooms)) AS baths
        WHERE
            ($community IS NULL OR area CONTAINS toLower($community))
            AND ($property_type IS NULL OR ptype = toLower($property_type))
            AND ($transaction_type IS NULL OR listing = toLower($transaction_type))
            AND ($status IS NULL OR state = toLower($status))
            AND ($min_price IS NULL OR price >= $min_price)
            AND ($max_price IS NULL OR price <= $max_price)
            AND ($min_beds IS NULL OR beds >= $min_beds)
            AND ($max_beds IS NULL OR beds <= $max_beds)
            AND ($min_bathrooms IS NULL OR baths >= $min_bathrooms)
            AND ($max_bathrooms IS NULL OR baths <= $max_bathrooms)
        RETURN
            toString(p.property_id) AS property_id,
            p.title AS title,
            p.price AS price,
            p.beds AS beds,
            p.bathrooms AS bathrooms,
            coalesce(c.name, p.community) AS community,
            coalesce(pt.name, p.property_type) AS property_type,
            p.listing_type AS listing_type,
            p.status AS status,
            p.description AS description,
            p.latitude AS latitude,
            p.longitude AS longitude
        LIMIT $limit
    """

    params = {
        "community": filters.get("community"),
        "property_type": filters.get("property_type"),
        "transaction_type": filters.get("transaction_type"),
        "status": filters.get("status"),
        "min_price": numeric_value(filters.get("min_price")),
        "max_price": numeric_value(filters.get("max_price")),
        "min_beds": numeric_value(filters.get("min_beds")),
        "max_beds": numeric_value(filters.get("max_beds")),
        "min_bathrooms": numeric_value(filters.get("min_bathrooms")),
        "max_bathrooms": numeric_value(filters.get("max_bathrooms")),
        "limit": int(limit),
    }
    return run_cypher(query, params)


def exact_property_lookup(property_id):
    pid = normalize_id(property_id)
    if not pid:
        return []
    return neo4j_records_by_ids([pid])


def community_analytics(community):
    return run_cypher("""
        MATCH (p:Property)-[:LOCATED_IN]->(c:Community)
        WHERE toLower(toString(c.name)) CONTAINS toLower($community)
        WITH c, p,
            toFloat(replace(toString(p.price), ',', '')) AS price
        RETURN
            c.name AS community,
            count(p) AS listing_count,
            avg(price) AS average_price,
            min(price) AS minimum_price,
            max(price) AS maximum_price
    """, {"community": community})


def retrieve_candidates(plan, top_k=TOP_K):
    strategy = plan["retrieval_strategy"]
    filters = plan.get("filters") or {}
    diagnostics = {"strategy": strategy}

    if plan.get("clarification_needed"):
        return [], diagnostics

    if strategy == "analytics":
        community = plan.get("community") or filters.get("community")
        if community:
            rows = community_analytics(community)
            diagnostics["analytics_rows"] = len(rows)
            return rows, diagnostics

        rows = run_cypher("""
            MATCH (p:Property)
            RETURN
                count(p) AS listing_count,
                avg(toFloat(replace(toString(p.price), ',', ''))) AS average_price,
                min(toFloat(replace(toString(p.price), ',', ''))) AS minimum_price,
                max(toFloat(replace(toString(p.price), ',', ''))) AS maximum_price
        """)
        diagnostics["analytics_rows"] = len(rows)
        return rows, diagnostics

    if strategy == "semantic":
        hits = milvus_search_ids(
            plan.get("semantic_query") or plan["normalized_query"]
        )
        ids = [hit["property_id"] for hit in hits]
        records = neo4j_records_by_ids(ids)
        diagnostics["milvus_hits"] = len(hits)
        diagnostics["neo4j_records"] = len(records)
        candidates = apply_hard_filters(records, filters)

    elif strategy == "structured":
        if plan.get("exact_property_id"):
            records = exact_property_lookup(plan["exact_property_id"])
        else:
            records = structured_search(filters, limit=1000)
        diagnostics["structured_records"] = len(records)
        candidates = apply_hard_filters(records, filters)

    elif strategy == "hybrid":
        # First establish eligible properties using explicit constraints.
        eligible = apply_hard_filters(
            structured_search(filters, limit=2000), filters
        )
        eligible_by_id = {
            normalize_id(r.get("property_id")): r
            for r in eligible if normalize_id(r.get("property_id"))
        }

        hits = milvus_search_ids(
            plan.get("semantic_query") or plan["normalized_query"]
        )
        hit_by_id = {
            h["property_id"]: h for h in hits
        }

        # Preserve semantic ranking, but never return an ineligible property.
        candidates = [
            eligible_by_id[pid]
            for pid in hit_by_id if pid in eligible_by_id
        ]

        # If semantic retrieval found no eligible match, return the
        # structured eligible set rather than incorrectly returning nothing.
        if not candidates:
            candidates = eligible

        diagnostics["eligible_records"] = len(eligible)
        diagnostics["milvus_hits"] = len(hits)
        diagnostics["semantic_eligible_matches"] = sum(
            pid in eligible_by_id for pid in hit_by_id
        )

    else:
        raise ValueError(f"Unsupported strategy: {strategy}")

    diagnostics["before_enrichment"] = len(candidates)
    candidates = enrich_with_csv(candidates)

    # Put semantic matches first when semantic scores are available.
    if strategy in {"semantic", "hybrid"}:
        hit_order = {
            h["property_id"]: i
            for i, h in enumerate(
                milvus_search_ids(
                    plan.get("semantic_query") or plan["normalized_query"]
                )
            )
        }
        candidates.sort(
            key=lambda row: hit_order.get(
                normalize_id(row.get("property_id")), 10**9
            )
        )

    diagnostics["after_enrichment"] = len(candidates)
    return candidates[:top_k], diagnostics

## 7. Grounded answer generation

In [12]:
ANSWER_PROMPT = """
You are a careful real-estate research assistant.

Answer using ONLY the supplied evidence.
Do not invent or infer missing property facts.
If no matching listings are available, say so clearly.
Distinguish facts from recommendations.
Use simple, readable language.
For each property, mention its ID, price, bedrooms, bathrooms, property
type, and community only when those fields are available.
Do not fabricate URLs or image links.
"""


def generate_answer(question, plan, records):
    if plan.get("clarification_needed"):
        return plan.get("clarification_question") or (
            "Could you clarify what kind of property you are looking for?"
        )

    if not records:
        return (
            "I couldn't find matching records in the connected data. "
            "Try adjusting the filters or broadening the search."
        )

    evidence = []
    for row in records:
        evidence.append({
            "property_id": row.get("property_id"),
            "title": row.get("title"),
            "price": row.get("price"),
            "beds": row.get("beds"),
            "bathrooms": row.get("bathrooms"),
            "community": row.get("community"),
            "property_type": row.get("property_type"),
            "listing_type": row.get("listing_type"),
            "status": row.get("status"),
            "description": row.get("description"),
            "building": row.get("building"),
        })

    response = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1800,
        system=ANSWER_PROMPT,
        messages=[{
            "role": "user",
            "content": (
                f"Question: {question}\n\n"
                f"Normalized plan:\n{json.dumps(plan, default=str)}\n\n"
                f"Retrieved evidence:\n{json.dumps(evidence, default=str)}"
            ),
        }],
    )

    return "".join(
        block.text for block in response.content
        if getattr(block, "type", None) == "text"
    )


def search_properties(question, top_k=TOP_K):
    plan = plan_query(question)
    records, diagnostics = retrieve_candidates(plan, top_k=top_k)
    answer = generate_answer(question, plan, records)

    return {
        "question": question,
        "plan": plan,
        "answer": answer,
        "records": records,
        "diagnostics": diagnostics,
    }

## 8. testing 

In [13]:
test_queries = [
    "Show apartments in Dubai Marina under 2.5 million AED",
    "Find a peaceful home with lots of natural light",
    "Find family-friendly apartments in Dubai Marina under 3 million AED",
    "How many properties are listed in Dubai Marina, and what is the average price?",
]

for question in test_queries:
    print("\\n" + "=" * 90)
    print("QUESTION:", question)
    try:
        result = search_properties(question)
        print("\\nPLAN:")
        print(json.dumps(result["plan"], indent=2, default=str))
        print("\\nANSWER:")
        print(result["answer"])
        print("\\nDIAGNOSTICS:")
        print(json.dumps(result["diagnostics"], indent=2, default=str))
        print("Returned listings:", len(result["records"]))
    except Exception as exc:
        print("Query failed:", repr(exc))

\n==========================================================================================
QUESTION: Show apartments in Dubai Marina under 2.5 million AED
\nPLAN:
{
  "query_type": "property_search",
  "retrieval_strategy": "structured",
  "normalized_query": "Find apartments in Dubai Marina with maximum price 2,500,000 AED",
  "semantic_query": null,
  "filters": {
    "community": "Dubai Marina",
    "property_type": "apartment",
    "transaction_type": "purchase",
    "status": "For Sale",
    "min_price": null,
    "max_price": 2500000,
    "min_beds": null,
    "max_beds": null,
    "min_bathrooms": null,
    "max_bathrooms": null
  },
  "exact_property_id": null,
  "community": "Dubai Marina",
  "intent_preferences": [],
  "answer_mode": "listings",
  "clarification_needed": false,
  "clarification_question": null
}
\nANSWER:
# Apartments in Dubai Marina Under 2.5 Million AED

I found 8 apartments matching your criteria:

---

## 1. **Property ID: 13947**
- **Price:** 2,000,000

## 9. Display property cards, images, and listing links

In [ ]:
def results_dataframe(records):
    rows = []
    for r in records:
        rows.append({
            "property_id": r.get("property_id"),
            "title": r.get("title"),
            "price": r.get("price"),
            "beds": r.get("beds"),
            "bathrooms": r.get("bathrooms"),
            "community": r.get("community"),
            "property_type": r.get("property_type"),
            "listing_type": r.get("listing_type"),
            "listing_url": r.get("listing_url"),
            "image_count": len(r.get("image_urls") or []),
        })
    return pd.DataFrame(rows)


def gallery_items(records):
    items = []
    for r in records:
        urls = r.get("image_urls") or []
        caption = (
            f"ID {r.get('property_id')} | "
            f"{r.get('title') or 'Property'} | "
            f"Price: {r.get('price') or 'Not available'}"
        )
        for url in urls[:3]:
            items.append((url, caption))
    return items


demo = search_properties(
    "Show apartments in Dubai Marina under 2.5 million AED"
)
print(demo["answer"])
display(results_dataframe(demo["records"]))
print("Gallery images:", len(gallery_items(demo["records"])))

# Apartments in Dubai Marina Under 2.5 Million AED

I found 8 apartments matching your criteria:

---

**1. Property ID: 13947**
- **Price:** 2,000,000 AED
- **Bedrooms:** 1 | **Bathrooms:** 2
- **Property Type:** Apartment
- **Community:** Dubai Marina
- Details: Spacious 1-bedroom with balcony and Marina views in Park Island. Features en-suite bathroom, built-in wardrobes, and extra storage. Located directly on Marina Walk with access to JBR beach and tram stops.

---

**2. Property ID: 32667**
- **Price:** 1,650,000 AED
- **Bedrooms:** 1 | **Bathrooms:** 2
- **Property Type:** Apartment
- **Community:** Dubai Marina
- Details: Fully furnished 1-bedroom with spacious layout, floor-to-ceiling windows, and modern finishes. Features open-plan living/dining area, built-in wardrobes, and fully equipped kitchen.

---

**3. Property ID: 92720**
- **Price:** 2,150,000 AED
- **Bedrooms:** 1 | **Bathrooms:** 2
- **Property Type:** Apartment
- **Community:** Dubai Marina
- Details: Bright 1-bed

,property_id,title,price,beds,bathrooms,community,property_type,listing_type,listing_url,image_count
0,13947,1 bedroom apartment for sale,2000000.0,1,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,13
1,32667,1 bedroom apartment for sale,1650000.0,1,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,33
2,92720,1 bedroom apartment for sale,2150000.0,1,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,18
3,94826,1 bedroom apartment for sale,2342000.0,1,1,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,38
4,94833,apartment for sale,1610000.0,0,1,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,39
5,103257,apartment for sale,2000000.0,0,1,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,14
6,122685,1 bedroom apartment for sale,1850000.0,1,2,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,12
7,131647,apartment for sale,1210000.0,0,1,Dubai Marina,apartment,purchase,https://www.hausandhaus.com/properties/for-sal...,13


Gallery images: 24


## 10. Gradio interface

In [15]:
import gradio as gr


def gradio_search(question):
    if not question or not question.strip():
        return "Enter a property question.", pd.DataFrame(), []

    try:
        result = search_properties(question.strip())
        table = results_dataframe(result["records"])
        gallery = gallery_items(result["records"])

        answer = result["answer"]
        answer += "\n\n---\n**Retrieval diagnostics**\n```json\n"
        answer += json.dumps(
            result["diagnostics"], indent=2, default=str
        )
        answer += "\n```"

        if not result["records"]:
            answer += (
                "\n\nNo property cards were returned. "
                "Check the planner and diagnostics above."
            )

        return answer, table, gallery

    except Exception as exc:
        return (
            f"Search failed: {type(exc).__name__}: {exc}",
            pd.DataFrame(),
            [],
        )


demo_app = gr.Interface(
    fn=gradio_search,
    inputs=gr.Textbox(
        label="Ask about properties",
        placeholder=(
            "e.g. Find family-friendly apartments in Dubai Marina "
            "under 3 million AED"
        ),
        lines=2,
    ),
    outputs=[
        gr.Markdown(label="Answer and diagnostics"),
        gr.Dataframe(label="Property results", wrap=True),
        gr.Gallery(
            label="Property images",
            columns=3,
            height="auto",
            object_fit="contain",
        ),
    ],
    title="Real Estate Research Assistant",
    description=(
        "Search property records using semantic retrieval, structured "
        "filters, or a combination of both."
    ),
    examples=[
        ["Show apartments in Dubai Marina under 2.5 million AED"],
        ["Find a peaceful home with lots of natural light"],
        ["Find family-friendly apartments in Dubai Marina"],
        ["How many properties are listed in Dubai Marina?"],
    ],
)

# Set share=True only if you intentionally want a public temporary URL.
demo_app.launch(share=False)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
i need an apartment which is having black chairs or black bathing tub and my dog should be allowed in that building.  Also there should be rain shower applied in the bathroom